# **TFM Project - Machine Learning for Drug Discovery in Neurodegenerative Diseases**
# **[Part 6] Optimisation of classification models**

Carla D. Di Monno

In **Part 6**, performs hyperparameter optimization and classification model evaluation for selectedtherapeutic targets, using algorithms like XGBoost, Random Forest, LightGBM, and LDA.

---

In [2]:
!pip install optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, matthews_corrcoef, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, make_scorer
)

# Model Imports
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# --- REPRODUCIBILITY CONFIGURATION ---
# Fixed seed for data splitting, cross-validation, and model initialization
SEED = 42

# --- TARGET TO MODEL MAPPING ---
# Assigning specific algorithms to each therapeutic target as requested
target_model_map = {
    "AChE.csv": "XGB",
    "MAO-B.csv": "RF",
    "GSK3B.csv": "LGBM",
    "LRRK2.csv": "LGBM",
    "LRRK2_WT.csv": "LGBM",
    "LRRK2_G2019S.csv": "LDA"
}

def objective(trial, X, y, model_name):
    """
    Optuna objective function for hyperparameter optimization.
    Returns the mean MCC score from a 5-fold cross-validation.
    """
    # Fix KFold random state for reproducible internal splits
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

    if model_name == "XGB":
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 800),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'random_state': SEED,
            'n_jobs': -1
        }
        model = XGBClassifier(**params)

    elif model_name == "RF":
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'max_depth': trial.suggest_int('max_depth', 5, 35),
            'random_state': SEED,
            'n_jobs': -1
        }
        model = RandomForestClassifier(**params)

    elif model_name == "LGBM":
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500),
            'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'random_state': SEED,
            'n_jobs': -1,
            'verbosity': -1
        }
        model = LGBMClassifier(**params)

    elif model_name == "LDA":
        params = {
            'solver': trial.suggest_categorical('solver', ['lsqr', 'eigen'])
        }
        # LDA shrinkage works with lsqr and eigen solvers
        params['shrinkage'] = trial.suggest_float('shrinkage', 0.0, 1.0)
        model = LDA(**params)

    # Use MCC as the scoring metric for optimization
    mcc_scorer = make_scorer(matthews_corrcoef)
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])

    # Calculate mean cross-validation score
    score = cross_val_score(pipeline, X, y, cv=kf, scoring=mcc_scorer, n_jobs=-1).mean()
    return score

# --- MAIN PROCESSING LOOP ---
summary_data = []

for ds_file in target_model_map.keys():
    print(f"\n" + "═"*60)
    print(f"🚀 Processing Dataset: {ds_file} | Model: {target_model_map[ds_file]}")
    print("═"*60)

    # Data Loading
    df = pd.read_csv(ds_file)
    # Drop regression target and class column to isolate features
    X = df.drop(columns=['bioactivity_class', 'pchembl_value'])
    # Encode target labels (Active/Inactive) into integers
    le = LabelEncoder()
    y = le.fit_transform(df['bioactivity_class'])

    # Train/Test Split (No stratification to match previous comparative evaluation)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED
    )

    m_name = target_model_map[ds_file]

    # Reproducible Optuna Study using a fixed-seed TPESampler
    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(lambda trial: objective(trial, X_train, y_train, m_name), n_trials=30)

    # Re-train Final Model with best parameters found
    bp = study.best_params
    print(f"  Best Parameters: {bp}")

    if m_name == "XGB": final_m = XGBClassifier(**bp, random_state=SEED)
    elif m_name == "RF": final_m = RandomForestClassifier(**bp, random_state=SEED)
    elif m_name == "LGBM": final_m = LGBMClassifier(**bp, random_state=SEED)
    elif m_name == "LDA": final_m = LDA(**bp)

    # Build and fit final pipeline
    final_pipe = Pipeline([('s', StandardScaler()), ('m', final_m)])
    final_pipe.fit(X_train, y_train)

    # Model Evaluation
    preds = final_pipe.predict(X_test)
    # Calculate probabilities for ROC-AUC
    if len(le.classes_) == 2:
        probs = final_pipe.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, probs)
    else:
        probs = final_pipe.predict_proba(X_test)
        auc = roc_auc_score(y_test, probs, multi_class='ovr')

    mcc = matthews_corrcoef(y_test, preds)
    acc = accuracy_score(y_test, preds)

    # Store and print results
    summary_data.append({"Target": ds_file, "Algorithm": m_name, "MCC": mcc, "ROC-AUC": auc, "Accuracy": acc})
    print(f"📈 Performance Metrics -> MCC: {mcc:.3f} | ROC-AUC: {auc:.3f} | Accuracy: {acc:.3f}")

    # Generate Confusion Matrix Plot
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, preds, display_labels=le.classes_,
                                            cmap='magma', ax=ax, colorbar=False)
    ax.set_title(f"{ds_file} ({m_name})\nMCC: {mcc:.3f} | AUC: {auc:.3f}")
    plt.tight_layout()
    plt.savefig(f"final_classification_{ds_file.replace('.csv', '')}.png", dpi=300)
    plt.close()

# Export Final Summary Table
df_summary = pd.DataFrame(summary_data)
print("\n" + "█"*30 + " FINAL EVALUATION SUMMARY " + "█"*30)
print(df_summary.to_string(index=False))
print("█"*86)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 11.1 MB/s eta 0:00:00

════════════════════════════════════════════════════════════
🚀 Processing Dataset: AChE.csv | Model: XGB
════════════════════════════════════════════════════════════


[I 2026-05-12 22:33:44,344] A new study created in memory with name: no-name-d158914f-639e-410d-ba6c-1862627102cc
[I 2026-05-12 22:34:01,594] Trial 0 finished with value: 0.6635118936531639 and parameters: {'n_estimators': 362, 'max_depth': 10, 'learning_rate': 0.08960785365368121}. Best is trial 0 with value: 0.6635118936531639.
[I 2026-05-12 22:34:13,375] Trial 1 finished with value: 0.5893212215315269 and parameters: {'n_estimators': 519, 'max_depth': 4, 'learning_rate': 0.015957084694148364}. Best is trial 0 with value: 0.6635118936531639.
[I 2026-05-12 22:34:19,928] Trial 2 finished with value: 0.6408595399038245 and parameters: {'n_estimators': 140, 'max_depth': 9, 'learning_rate': 0.06054365855469246}. Best is trial 0 with value: 0.6635118936531639.
[I 2026-05-12 22:34:30,522] Trial 3 finished with value: 0.638589207080636 and parameters: {'n_estimators': 596, 'max_depth': 3, 'learning_rate': 0.18276027831785724}. Best is trial 0 with value: 0.6635118936531639.
[I 2026-05-12 22:

  Best Parameters: {'n_estimators': 365, 'max_depth': 9, 'learning_rate': 0.118612417644104}
📈 Performance Metrics -> MCC: 0.649 | ROC-AUC: 0.901 | Accuracy: 0.826


[I 2026-05-12 22:41:17,958] A new study created in memory with name: no-name-d9056b88-c530-49f6-96d9-769f5da6f91e



════════════════════════════════════════════════════════════
🚀 Processing Dataset: MAO-B.csv | Model: RF
════════════════════════════════════════════════════════════


...

  Best Parameters: {'n_estimators': 102, 'num_leaves': 98, 'learning_rate': 0.11851057563919334}
[LightGBM] [Info] Number of positive: 905, number of negative: 132
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004063 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1125
[LightGBM] [Info] Number of data points in the train set: 1037, number of used features: 375
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.872710 -> initscore=1.925133
[LightGBM] [Info] Start training from score 1.925133
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positiv

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-05-12 23:12:56,122] A new study created in memory with name: no-name-329c463d-9149-407f-873d-6601e5da66eb



════════════════════════════════════════════════════════════
🚀 Processing Dataset: LRRK2_G2019S.csv | Model: LDA
════════════════════════════════════════════════════════════


[I 2026-05-12 23:12:56,821] Trial 0 finished with value: 0.40220348267946626 and parameters: {'solver': 'eigen', 'shrinkage': 0.7319939418114051}. Best is trial 0 with value: 0.40220348267946626.
[I 2026-05-12 23:12:57,459] Trial 1 finished with value: 0.40351856342855497 and parameters: {'solver': 'lsqr', 'shrinkage': 0.15599452033620265}. Best is trial 1 with value: 0.40351856342855497.
[I 2026-05-12 23:12:58,139] Trial 2 finished with value: 0.4062470453358009 and parameters: {'solver': 'eigen', 'shrinkage': 0.6011150117432088}. Best is trial 2 with value: 0.4062470453358009.
[I 2026-05-12 23:12:58,736] Trial 3 finished with value: 0.3250697549922224 and parameters: {'solver': 'lsqr', 'shrinkage': 0.9699098521619943}. Best is trial 2 with value: 0.4062470453358009.
[I 2026-05-12 23:12:59,362] Trial 4 finished with value: 0.40895567566592417 and parameters: {'solver': 'lsqr', 'shrinkage': 0.18182496720710062}. Best is trial 4 with value: 0.40895567566592417.
[I 2026-05-12 23:13:00,02

  Best Parameters: {'solver': 'eigen', 'shrinkage': 0.003617695313975546}
📈 Performance Metrics -> MCC: 0.340 | ROC-AUC: 0.780 | Accuracy: 0.897

██████████████████████████████ FINAL EVALUATION SUMMARY ██████████████████████████████
          Target Algorithm      MCC  ROC-AUC  Accuracy
        AChE.csv       XGB 0.649211 0.901366  0.825737
       MAO-B.csv        RF 0.642341 0.896138  0.822323
       GSK3B.csv      LGBM 0.640507 0.906088  0.834426
       LRRK2.csv      LGBM 0.497381 0.886775  0.913580
    LRRK2_WT.csv      LGBM 0.333635 0.866460  0.830769
LRRK2_G2019S.csv       LDA 0.339785 0.779892  0.897281
██████████████████████████████████████████████████████████████████████████████████████
